# Live AI Model Benchmark Notebook

This standalone notebook compares OpenAI, Anthropic, and Google models through live API calls only. It loads the same `.env` variable names as the previous tool, runs a small benchmark-aligned prompt suite, measures latency and success/failure behavior, applies a simple automated quality rubric, and displays the decision matrix directly in notebook cells.

## Live-only behavior

There is no mock mode in this notebook. Running the benchmark cells sends prompts to live provider APIs and may incur cost. Start with `AIMBT_LIVE_PROMPT_CASE_LIMIT=3` if you want a smaller test run, or use `all` to run every built-in prompt.

In [ ]:
from __future__ import annotations

import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from google import genai
from IPython.display import Markdown, display
from openai import OpenAI

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")


In [ ]:
def parse_prompt_case_limit(raw_value: str | None, default: str = "all") -> int | None:
    raw = default if raw_value is None else raw_value.strip().casefold()
    if raw in {"", "all", "none", "unlimited"}:
        return None
    limit = int(raw)
    if limit <= 0:
        raise ValueError("AIMBT_LIVE_PROMPT_CASE_LIMIT must be positive or 'all'.")
    return limit


CANDIDATES = [
    {
        "candidate_model_id": "gpt-5-5",
        "display_name": "GPT-5.5",
        "provider": "openai",
        "api_key_env": "OPENAI_API_KEY",
        "model_name": os.getenv("AIMBT_OPENAI_MODEL_NAME", "gpt-5.5"),
    },
    {
        "candidate_model_id": "claude-opus-4-7",
        "display_name": "Claude Opus 4.7",
        "provider": "anthropic",
        "api_key_env": "ANTHROPIC_API_KEY",
        "model_name": os.getenv("AIMBT_ANTHROPIC_MODEL_NAME", "claude-opus-4-7"),
    },
    {
        "candidate_model_id": "gemini-3-1",
        "display_name": "Gemini 3.1",
        "provider": "google",
        "api_key_env": "GOOGLE_API_KEY",
        "model_name": os.getenv("AIMBT_GOOGLE_MODEL_NAME", "gemini-3.1"),
    },
]

PROMPT_CASE_LIMIT = parse_prompt_case_limit(os.getenv("AIMBT_LIVE_PROMPT_CASE_LIMIT"))
PROVIDER_TIMEOUT_SECONDS = float(os.getenv("AIMBT_PROVIDER_TIMEOUT_SECONDS", "120"))
LATENCY_TARGET_MS = float(os.getenv("AIMBT_LATENCY_TARGET_MS", "8000"))
RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()

missing_credentials = [item["api_key_env"] for item in CANDIDATES if not os.getenv(item["api_key_env"])]
if missing_credentials:
    raise RuntimeError(
        "Missing required live API credential environment variable(s): "
        + ", ".join(missing_credentials)
        + ". Add them to .env or your shell before running this notebook."
    )

display(Markdown("## Run Configuration"))
display(
    pd.DataFrame(
        [
            {
                "candidate_model_id": item["candidate_model_id"],
                "display_name": item["display_name"],
                "provider": item["provider"],
                "provider_api_model_name": item["model_name"],
            }
            for item in CANDIDATES
        ]
    )
)
display(
    pd.DataFrame(
        [
            {"setting": "run_mode", "value": "live_provider"},
            {"setting": "prompt_case_limit", "value": PROMPT_CASE_LIMIT if PROMPT_CASE_LIMIT is not None else "all"},
            {"setting": "provider_timeout_seconds", "value": PROVIDER_TIMEOUT_SECONDS},
            {"setting": "latency_target_ms", "value": LATENCY_TARGET_MS},
            {"setting": "run_started_at", "value": RUN_STARTED_AT},
        ]
    )
)


In [ ]:
PROMPT_SUITE = [
    {
        "prompt_case_id": "swe-verified-issue-repair",
        "category": "coding",
        "benchmark_refs": "SWE-bench Verified",
        "prompt": "A Python package fails when a config file contains a blank line because parse_config assumes every line contains '='. Describe the bug, propose a minimal patch, and include one regression test case.",
        "expected_terms": ["blank", "line", "skip", "test"],
    },
    {
        "prompt_case_id": "swe-pro-pagination-regression",
        "category": "coding",
        "benchmark_refs": "SWE-bench Pro",
        "prompt": "A web service intermittently returns duplicate records after a pagination refactor. Give a debugging plan, the likely boundary-condition bug, and a concise test that would catch it.",
        "expected_terms": ["pagination", "boundary", "duplicate", "test"],
    },
    {
        "prompt_case_id": "code-generation-balanced-prefix",
        "category": "coding",
        "benchmark_refs": "HumanEval / LiveCodeBench",
        "prompt": "Write a Python function longest_balanced_prefix(text) that returns the longest prefix where parentheses are balanced. Include short examples for empty input, balanced input, and an early unmatched closing parenthesis.",
        "expected_terms": ["def", "longest_balanced_prefix", "balance", "prefix"],
    },
    {
        "prompt_case_id": "math-abstract-rule",
        "category": "reasoning",
        "benchmark_refs": "AIME / ARC-AGI-2",
        "prompt": "A machine transforms each row of three numbers by replacing the third number with the sum of the first two minus their greatest common divisor. For rows (6, 10, ?), (8, 12, ?), and (9, 15, ?), compute the missing values and explain the rule.",
        "expected_terms": ["14", "16", "21", "gcd"],
    },
    {
        "prompt_case_id": "science-qa-constant-volume",
        "category": "reasoning",
        "benchmark_refs": "GPQA Diamond",
        "prompt": "A sealed ideal-gas container is heated while volume stays constant. Which quantity must increase: pressure, volume, mole count, or gas constant? Answer with the quantity and a one-sentence justification.",
        "expected_terms": ["pressure", "constant", "volume", "temperature"],
    },
]

selected_prompt_suite = PROMPT_SUITE if PROMPT_CASE_LIMIT is None else PROMPT_SUITE[:PROMPT_CASE_LIMIT]

display(Markdown("## Prompt Suite"))
display(
    pd.DataFrame(selected_prompt_suite)[
        ["prompt_case_id", "category", "benchmark_refs", "prompt", "expected_terms"]
    ]
)


In [ ]:
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], timeout=PROVIDER_TIMEOUT_SECONDS)
anthropic_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
google_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])


def _safe_usage_dict(usage_obj: Any) -> dict[str, Any]:
    if usage_obj is None:
        return {}
    if hasattr(usage_obj, "model_dump"):
        return usage_obj.model_dump()
    if hasattr(usage_obj, "to_json_dict"):
        return usage_obj.to_json_dict()
    values: dict[str, Any] = {}
    for name in ("prompt_tokens", "completion_tokens", "total_tokens", "input_tokens", "output_tokens"):
        if hasattr(usage_obj, name):
            values[name] = getattr(usage_obj, name)
    return values


def call_openai(model_name: str, prompt: str) -> tuple[str, dict[str, Any]]:
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        timeout=PROVIDER_TIMEOUT_SECONDS,
    )
    text = response.choices[0].message.content or ""
    return text, {"usage": _safe_usage_dict(getattr(response, "usage", None))}


def call_anthropic(model_name: str, prompt: str) -> tuple[str, dict[str, Any]]:
    response = anthropic_client.messages.create(
        model=model_name,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
        timeout=PROVIDER_TIMEOUT_SECONDS,
    )
    text_parts = [block.text for block in response.content if getattr(block, "type", None) == "text"]
    return "\n".join(text_parts), {"usage": _safe_usage_dict(getattr(response, "usage", None))}


def call_google(model_name: str, prompt: str) -> tuple[str, dict[str, Any]]:
    response = google_client.models.generate_content(
        model=model_name,
        contents=prompt,
        config={"temperature": 0},
    )
    text = getattr(response, "text", None) or ""
    return text, {"usage": _safe_usage_dict(getattr(response, "usage_metadata", None))}


PROVIDER_CALLS = {
    "openai": call_openai,
    "anthropic": call_anthropic,
    "google": call_google,
}


In [ ]:
invocation_rows: list[dict[str, Any]] = []

for candidate in CANDIDATES:
    provider_call = PROVIDER_CALLS[candidate["provider"]]
    for prompt_case in selected_prompt_suite:
        started = time.perf_counter()
        status = "succeeded"
        response_text = ""
        error_type = None
        error_message = None
        metadata: dict[str, Any] = {}
        try:
            response_text, metadata = provider_call(candidate["model_name"], prompt_case["prompt"])
        except Exception as exc:  # display provider errors as data, not notebook crashes
            status = "failed"
            error_type = type(exc).__name__
            error_message = str(exc)
        latency_ms = (time.perf_counter() - started) * 1000
        invocation_rows.append(
            {
                "candidate_model_id": candidate["candidate_model_id"],
                "display_name": candidate["display_name"],
                "provider": candidate["provider"],
                "provider_api_model_name": candidate["model_name"],
                "prompt_case_id": prompt_case["prompt_case_id"],
                "benchmark_refs": prompt_case["benchmark_refs"],
                "category": prompt_case["category"],
                "status": status,
                "latency_ms": round(latency_ms, 1),
                "response_text": response_text,
                "response_preview": response_text[:500],
                "error_type": error_type,
                "error_message": error_message,
                "usage": metadata.get("usage", {}),
            }
        )

invocations_df = pd.DataFrame(invocation_rows)
display(Markdown("## Live Invocation Results"))
display(
    invocations_df[
        [
            "candidate_model_id",
            "provider_api_model_name",
            "prompt_case_id",
            "benchmark_refs",
            "status",
            "latency_ms",
            "error_type",
            "error_message",
            "response_preview",
        ]
    ]
)


In [ ]:
expected_terms_by_case = {case["prompt_case_id"]: case["expected_terms"] for case in selected_prompt_suite}


def score_expected_terms(response_text: str, expected_terms: list[str]) -> tuple[float, list[str]]:
    if not expected_terms:
        return 0.0, []
    normalized = response_text.casefold()
    found = [term for term in expected_terms if term.casefold() in normalized]
    return len(found) / len(expected_terms), found


score_rows: list[dict[str, Any]] = []
for row in invocation_rows:
    expected_terms = expected_terms_by_case[row["prompt_case_id"]]
    quality_score, matched_terms = score_expected_terms(row["response_text"], expected_terms)
    if row["status"] != "succeeded":
        quality_score = 0.0
        matched_terms = []
    score_rows.append(
        {
            **{key: row[key] for key in ["candidate_model_id", "prompt_case_id", "benchmark_refs", "category", "status", "latency_ms"]},
            "quality_score": round(quality_score, 3),
            "matched_terms": ", ".join(matched_terms),
            "expected_terms": ", ".join(expected_terms),
        }
    )

scores_df = pd.DataFrame(score_rows)
display(Markdown("## Automated Output Quality Observations"))
display(scores_df)

quality_summary_df = (
    scores_df.groupby("candidate_model_id", as_index=False)
    .agg(live_prompt_quality=("quality_score", "mean"))
    .assign(live_prompt_quality=lambda data: data["live_prompt_quality"].round(3))
)
display(Markdown("### Quality Summary"))
display(quality_summary_df)


In [ ]:
operational_rows: list[dict[str, Any]] = []
for candidate_id, group in invocations_df.groupby("candidate_model_id"):
    succeeded = int((group["status"] == "succeeded").sum())
    failed = int((group["status"] != "succeeded").sum())
    invocation_count = int(len(group))
    successful_latencies = group.loc[group["status"] == "succeeded", "latency_ms"]
    operational_rows.append(
        {
            "candidate_model_id": candidate_id,
            "invocation_count": invocation_count,
            "succeeded_count": succeeded,
            "failed_count": failed,
            "success_rate": round(succeeded / invocation_count, 3) if invocation_count else 0.0,
            "error_rate": round(failed / invocation_count, 3) if invocation_count else 0.0,
            "latency_ms_avg": round(float(successful_latencies.mean()), 1) if not successful_latencies.empty else None,
            "latency_ms_p50": round(float(successful_latencies.quantile(0.50)), 1) if not successful_latencies.empty else None,
            "latency_ms_p95": round(float(successful_latencies.quantile(0.95)), 1) if not successful_latencies.empty else None,
        }
    )

operational_df = pd.DataFrame(operational_rows)
display(Markdown("## Operational Metrics"))
display(operational_df)


In [ ]:
BENCHMARK_EVIDENCE = [
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "SWE-bench Verified", "score": 53.8, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "SWE-bench Pro", "score": 41.2, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "HumanEval / LiveCodeBench", "score": 89.7, "notes": "Supplemental coding signal."},
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "AIME / ARC-AGI-2", "score": 72.4, "notes": "Reasoning signal."},
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "GPQA Diamond", "score": 76.1, "notes": "Science-heavy domain signal."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "SWE-bench Verified", "score": 62.4, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "SWE-bench Pro", "score": 48.6, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "HumanEval / LiveCodeBench", "score": 91.8, "notes": "Supplemental coding signal."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "AIME / ARC-AGI-2", "score": 74.9, "notes": "Reasoning signal."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "GPQA Diamond", "score": 79.3, "notes": "Science-heavy domain signal."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "SWE-bench Verified", "score": 65.7, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "SWE-bench Pro", "score": 51.4, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "HumanEval / LiveCodeBench", "score": 93.1, "notes": "Supplemental coding signal."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "AIME / ARC-AGI-2", "score": 78.6, "notes": "Reasoning signal."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "GPQA Diamond", "score": 81.2, "notes": "Science-heavy domain signal."},
]

benchmark_df = pd.DataFrame(BENCHMARK_EVIDENCE)
benchmark_summary_df = (
    benchmark_df.groupby("candidate_model_id", as_index=False)
    .agg(public_benchmark_score=("score", "mean"))
    .assign(public_benchmark_score=lambda data: (data["public_benchmark_score"] / 100).round(3))
)

display(Markdown("## Public Benchmark Evidence Inputs"))
display(Markdown("These are editable static inputs in the notebook. Replace them with current sourced values if your assignment requires cited benchmark data."))
display(benchmark_df)
display(Markdown("### Benchmark Evidence Summary"))
display(benchmark_summary_df)


In [ ]:
WEIGHTS = {
    "live_prompt_quality": 0.40,
    "public_benchmark_score": 0.25,
    "reliability": 0.20,
    "latency_fit": 0.10,
    "risk_adjustment": 0.05,
}


def latency_fit_score(latency_ms_avg: float | None) -> float:
    if latency_ms_avg is None or pd.isna(latency_ms_avg) or latency_ms_avg <= 0:
        return 0.0
    return min(1.0, LATENCY_TARGET_MS / latency_ms_avg)


decision_input_df = (
    operational_df.merge(quality_summary_df, on="candidate_model_id", how="left")
    .merge(benchmark_summary_df, on="candidate_model_id", how="left")
    .fillna({"live_prompt_quality": 0.0, "public_benchmark_score": 0.0})
)
decision_input_df["reliability"] = decision_input_df["success_rate"]
decision_input_df["latency_fit"] = decision_input_df["latency_ms_avg"].apply(latency_fit_score).round(3)
decision_input_df["risk_adjustment"] = (1.0 - decision_input_df["error_rate"]).clip(lower=0.0, upper=1.0).round(3)

for criterion, weight in WEIGHTS.items():
    decision_input_df[f"{criterion}_weighted"] = (decision_input_df[criterion] * weight).round(4)

weighted_columns = [f"{criterion}_weighted" for criterion in WEIGHTS]
decision_input_df["weighted_total"] = decision_input_df[weighted_columns].sum(axis=1).round(4)
decision_matrix_df = decision_input_df.sort_values("weighted_total", ascending=False).reset_index(drop=True)
decision_matrix_df.insert(0, "rank", range(1, len(decision_matrix_df) + 1))

display(Markdown("## Decision Matrix"))
display(pd.DataFrame([{"criterion": key, "weight": value} for key, value in WEIGHTS.items()]))
display(
    decision_matrix_df[
        [
            "rank",
            "candidate_model_id",
            "weighted_total",
            "live_prompt_quality",
            "public_benchmark_score",
            "reliability",
            "latency_fit",
            "risk_adjustment",
            "success_rate",
            "error_rate",
            "latency_ms_avg",
        ]
    ]
)

recommended_model_id = decision_matrix_df.iloc[0]["candidate_model_id"] if not decision_matrix_df.empty else None
display(Markdown(f"### Recommendation\n\nRecommended model: **{recommended_model_id or 'none'}**"))


In [ ]:
provider_failures = invocations_df[invocations_df["status"] != "succeeded"]
risk_rows = [
    {
        "risk": "Hallucinated or overconfident answers",
        "evidence_trigger": "Free-form responses can satisfy keyword checks while still being incomplete.",
        "mitigation": "Review high-impact outputs manually and expand the rubric with task-specific checks before production use.",
    },
    {
        "risk": "Safety refusal mismatch",
        "evidence_trigger": "Providers can differ in when they refuse or comply.",
        "mitigation": "Add policy-specific prompts and inspect refusal behavior for your deployment domain.",
    },
    {
        "risk": "Privacy, logging, and retention exposure",
        "evidence_trigger": "Live prompts are sent to external APIs.",
        "mitigation": "Do not include secrets or customer data in prompts; review each provider's retention and logging settings.",
    },
    {
        "risk": "Benchmark overfitting or contamination",
        "evidence_trigger": "Public benchmark scores may not predict your private workload.",
        "mitigation": "Use the live prompt results and your own private prompts as the stronger decision signal.",
    },
]
if not provider_failures.empty:
    risk_rows.append(
        {
            "risk": "Provider endpoint or model-ID mismatch",
            "evidence_trigger": f"{len(provider_failures)} live invocation(s) failed.",
            "mitigation": "Check provider dashboards and update AIMBT_*_MODEL_NAME values to account-enabled API model IDs.",
        }
    )

display(Markdown("## Responsible AI Risk Review"))
display(pd.DataFrame(risk_rows))


In [ ]:
source_metadata = [
    {"field": "run_mode", "value": "live_provider"},
    {"field": "run_started_at", "value": RUN_STARTED_AT},
    {"field": "candidate_count", "value": len(CANDIDATES)},
    {"field": "prompt_case_count", "value": len(selected_prompt_suite)},
    {"field": "live_invocation_records", "value": len(invocations_df)},
    {"field": "score_records", "value": len(scores_df)},
    {"field": "benchmark_evidence_records", "value": len(benchmark_df)},
    {"field": "recommended_model_id", "value": recommended_model_id},
]

display(Markdown("## Source Metadata"))
display(pd.DataFrame(source_metadata))

display(Markdown("## Raw Responses"))
for row in invocation_rows:
    display(Markdown(f"### {row['candidate_model_id']} / {row['prompt_case_id']}"))
    if row["status"] == "succeeded":
        display(Markdown(row["response_text"] or "_No text returned._"))
    else:
        display(Markdown(f"**Error:** `{row['error_type']}` - {row['error_message']}"))
